# Final Pipeline Tutorial: Evidence Selection And Classification

This tutorial keeps the second-meeting story arc, but updates the methods and numbers to the Colab-feasible submitted pipeline.

```text
raw JSON
-> efficient sparse retrieval sources
-> Sparse fused pool: top500 candidates
-> Embedding + factual cues: top64 context
-> Cross-encoder + embedding: top3 submitted evidence
-> Enhanced Context20 TF-IDF logistic classifier
```

The main implementation change is computational, not conceptual: sparse retrieval uses sklearn/scipy matrix operators, top-k uses NumPy selection, and neural scoring is restricted to batched candidate subsets rather than Python-level dense claim-evidence loops.


# 2. Method Names Used In The Figures

The figures use display names rather than internal experiment IDs.

| display name | meaning | used for |
| --- | --- | --- |
| Word match | BM25-style lexical matching between claim words and evidence text | sparse retrieval source |
| Character match | character n-gram TF-IDF matching, useful when wording changes slightly | sparse retrieval source |
| Structured facts | cheap factual cues such as numbers, years, and explicit pattern matches | sparse retrieval source |
| Query expansion | sparse retrieval after adding related query terms | sparse retrieval source |
| Sparse fused pool | fixed weighted fusion of the four sparse sources | submitted top500 candidate pool |
| Embedding + factual cues | MiniLM sentence-embedding similarity plus compact overlap/rank/number features | top64 classifier context |
| Cross-encoder + embedding | bounded transformer pair scoring plus embedding score | final top3 evidence ranking |

Metrics are also named explicitly: `macro recall@N` averages evidence recall over claims, `evidence F@3` is the submitted evidence score, and `C@N` means the share of claims with at least one gold evidence item in the first `N` rows.


In [ ]:
# 1. Imports and plotting defaults
from pathlib import Path
import json
from IPython.display import Image, display

ROOT = Path.cwd()
if (ROOT / 'third_tutorial_metrics.json').exists():
    MEETING_DIR = ROOT
elif (ROOT / 'group_meetings/third_meeting ' / 'third_tutorial_metrics.json').exists():
    MEETING_DIR = ROOT / 'group_meetings/third_meeting '
else:
    MEETING_DIR = Path('group_meetings/third_meeting ')
FIG_DIR = MEETING_DIR / 'figures'

with (MEETING_DIR / 'third_tutorial_metrics.json').open(encoding='utf-8') as f:
    third = json.load(f)
summary = third['summary']
classifier = third['classifier']

def showfig(name, width=None):
    display(Image(filename=str(FIG_DIR / name), width=width))

def pct(x):
    return f'{x * 100:.1f}%'

# 2. Load Data And Helpers

All figures below are recomputed from the third-meeting Colab-feasible outputs or from the same raw dev labels. The data scale is unchanged from the second meeting; the method names, parameters, and scores are updated to match the submitted notebook.

In [ ]:
print('Final dev metrics')
print(json.dumps(summary['final_metrics'], indent=2))
print('Total wall time:', f"{summary['total_wall_seconds']:.1f}s")

# 3. Dataset And Retrieval Cost

The dataset summary remains a categorical comparison, so the original bar-chart form is kept. The cost chart is also still categorical; it now highlights the submitted implementation constraint: expensive neural scoring is bounded by sparse candidates and cross-encoder prefiltering.

In [ ]:
showfig('third_dataset_and_labels.png')
showfig('third_candidate_cost_bar.png')

# 4. Sparse Retrieval And Score Fusion

The four sparse retrieval sources are word matching, character matching, structured factual cues, and query expansion. We keep the R-N curve form to show how recall changes as the candidate budget grows.

The submitted top500 candidate pool is **Sparse fused pool**: a fixed weighted fusion of these four sources.


In [ ]:
showfig('third_single_sparse_rn.png')

Score fusion assigns fixed weights to the four sparse retrieval sources and produces one ranked top500 pool. The submitted weights are `word match=0.75`, `character match=2.0`, `structured facts=0.25`, and `query expansion=0.5`, with `RRF k=500`. The figure keeps the same three-part structure: weights, R-N curve, and top500 recall.


In [ ]:
showfig('third_sparse_fusion_weights_and_rn.png')
print('Submitted sparse top500:', summary['candidate'])

The final top500 candidate pool uses **Sparse fused pool** because it gives the required coverage while staying inside the Colab runtime budget. We do not score every claim-evidence pair. Embedding and cross-encoder scoring are reserved for smaller subsets after this sparse pool.


In [ ]:
showfig('third_cost_benefit_staging.png')

The submitted pipeline uses compact factual cues rather than a large offline feature bank. These cues include sparse retrieval rank, embedding score, word overlap, number overlap, length shape, and negation agreement. They are cheap enough to compute inside the Colab pipeline and dense enough to improve the top64 context.


In [ ]:
showfig('third_hand_feature_design.png')

# 7. Top-64 Context Selection

The final top64 context selector is **Embedding + factual cues**. This is the classifier-context operating point: it aims to put enough gold evidence into the first 64 rows before the classifier consumes an enhanced top20 view.

This comparison is restricted to methods designed for context selection: sparse cutoff, cross-encoder plus factual cues, and embedding plus factual cues. The later top3 fusion is not included here because it optimizes the submitted evidence list, not classifier context.


In [ ]:
showfig('third_top64_selector_bar.png')
print('Final top64 macro recall:', pct(summary['top64']['macro_recall@64']))

# 8. Factual Cues As A Complement

This ablation keeps the base ranker fixed and changes only whether factual cues are added. That is the right test for whether shallow features help: compare `embedding only` against `embedding + factual cues`, and compare `cross-encoder only` against `cross-encoder + factual cues`.

The result is intuitive. The embedding ranker uses a compressed sentence vector and an inner product, so it can blur exact facts such as numbers, negation, entity overlap, and short lexical matches. Adding factual cues restores some of that lost surface information, giving a larger gain near the top64 context cutoff: `58.0%` versus `56.5%` macro recall@64.

The cross-encoder already reads the claim and evidence together with token-level interactions, so it can directly model many of the same lexical and factual cues. The extra factual features therefore have a much smaller marginal effect: `56.7%` versus `56.4%` macro recall@64.


In [ ]:
showfig('third_shallow_complement_two_panel_rn.png')

# 9. Top-3 Evidence For Submission

The submitted evidence list is a separate operating point from classifier context. Top64 is for context coverage; top3 is the final evidence ranking submitted with each claim.

The submitted top3 ranker is **Cross-encoder + embedding**. The cross-encoder gives high-quality pair scores but is expensive, so it only scores a bounded subset. The train-selected fusion weights are `cross-encoder score=0.25`, `cross-encoder rank=0.25`, and `embedding score=0.50`; the other candidate signals receive zero weight in the selected top3 model.

The C@N curve below is diagnostic for the top3 ranking stage. It should not be read as the top64 context-selector decision.


In [ ]:
showfig('third_top3_gate_cn.png')
showfig('third_top3_submission_rn.png')
showfig('third_top3_submission_bar.png')
print('Final top3:', summary['top3'])

# 10. Classifier Hyperparameter Search

The retrieval and evidence-ranking stages are fixed before classification. The submitted classifier is the **Enhanced Context20 TF-IDF Logistic Classifier**:

| part | submitted configuration |
| --- | --- |
| context source | Embedding + factual cues top64 |
| context input | enhanced Context20 |
| top evidence budget | 120 tokens for top5 evidence |
| tail evidence budget | 50 tokens for ranks 6-20 |
| text representation | TF-IDF word `1-2` grams |
| max TF-IDF features | `60000` |
| classifier | one-vs-rest logistic regression |
| regularization | `C=0.125` |
| class weighting | balanced |

The enhanced input keeps the same evidence text but adds rank-aware and factual cue tokens, so the linear classifier can learn from shallow signals without hard-coded prediction rules.


In [ ]:
showfig('third_context20_classifier_result.png')
print('Classifier accuracy:', pct(classifier['accuracy']))
print('Classifier macro-F1:', pct(classifier['macro_f1']))
print('Prediction histogram:', classifier['prediction_histogram'])

# 11. Sensitivity Analysis

After the selected classifier result is fixed, the last section keeps the same two-panel sensitivity structure. The first curve varies logistic `C` inside the enhanced Context20 setup. The second curve varies the number of evidence snippets while holding the selected classifier family fixed.

In [ ]:
showfig('third_classifier_sensitivity.png')